In [1]:
import networkx as nx
import pandas as pd
from tqdm import tqdm
import reverse_geocoder as rg
from utils.connecticut_town_to_region import town_to_planning_region
import pickle

In [2]:
tqdm.pandas()

# Load Yale data

In [3]:
yale = pd.read_excel("YCOM_2024_publicdata.xlsx",sheet_name='YCOM8_2024')

In [4]:
yale.head()

,GeoType,GeoID,GeoName,population,affectweather,affectweatherOppose,attrdroughts,attrdroughtsOppose,attrflood,attrfloodOppose,...,regulate,regulateOppose,teachgw,teachgwOppose,timing,timingOppose,transitioneconomy,transitioneconomyOppose,worried,worriedOppose
0,national,9999,US,258742321,65.45,5.69,60.67,25.41,58.40,27.03,...,73.97,26.03,77.41,22.59,57.92,42.08,66.24,33.76,63.37,36.63
1,state,1,Alabama,3926251,54.68,7.02,46.71,32.53,45.30,34.64,...,69.13,30.87,73.90,26.10,49.88,50.12,56.51,43.49,53.53,46.47
2,state,2,Alaska,555513,64.08,6.05,61.43,27.37,54.67,31.83,...,69.86,30.14,74.77,25.23,54.19,45.81,58.11,41.89,59.17,40.83
3,state,4,Arizona,5673394,65.50,5.62,63.00,25.70,59.00,27.52,...,74.44,25.56,75.40,24.60,56.57,43.43,64.39,35.61,63.77,36.23
4,state,5,Arkansas,2326772,54.41,7.87,49.34,35.50,45.31,38.63,...,66.57,33.43,70.73,29.27,49.28,50.72,53.42,46.58,53.09,46.91


In [5]:
yale_2024_county = yale[(yale['GeoType'] == 'county')]

In [6]:
def separate_county_and_state(row):
    split = row["GeoName"].split(",")
    county = split[0].strip()
    state = split[1].strip()
    return pd.Series([county, state])
    
yale_2024_county[['county', 'state']] = yale_2024_county.progress_apply(separate_county_and_state, axis=1)

100%|██████████| 3144/3144 [00:00<00:00, 8880.46it/s] 
C:\Users\Elizabeth\AppData\Local\Temp\ipykernel_35836\1205319347.py:7: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  yale_2024_county[['county', 'state']] = yale_2024_county.progress_apply(separate_county_and_state, axis=1)
C:\Users\Elizabeth\AppData\Local\Temp\ipykernel_35836\1205319347.py:7: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  yale_2024_county[['county', 'state']] = yale_2024_county.progress_apply(separate_county_and_state, axis=1)


# Load Gowalla data

In [8]:
gowalla_total = pd.read_csv("gowalla_data/Gowalla_totalCheckins.txt",     
                             sep='\t', header=None)
gowalla_total.columns = ['userid','timestamp','latitude','longitude','spotid']
gowalla_total.head()

,userid,timestamp,latitude,longitude,spotid
0,0,2010-10-19T23:55:27Z,30.235909,-97.795140,22847
1,0,2010-10-18T22:17:43Z,30.269103,-97.749395,420315
2,0,2010-10-17T23:42:03Z,30.255731,-97.763386,316637
3,0,2010-10-17T19:26:05Z,30.263418,-97.757597,16516
4,0,2010-10-16T18:50:42Z,30.274292,-97.740523,5535878


## Get county for each entry in Gowalla data

In [9]:
coords = list(zip(gowalla_total['latitude'], gowalla_total['longitude']))

batch_size = 100000 
results = []

for i in tqdm(range(0, len(coords), batch_size)):
    chunk = coords[i:i+batch_size]
    res_chunk = rg.search(chunk, mode=2)
    results.extend(res_chunk)

  0%|          | 0/65 [00:00<?, ?it/s]

Loading formatted geocoded file...


100%|██████████| 65/65 [06:35<00:00,  6.09s/it]


In [11]:
gowalla_total.head()

,userid,timestamp,latitude,longitude,spotid
0,0,2010-10-19T23:55:27Z,30.235909,-97.795140,22847
1,0,2010-10-18T22:17:43Z,30.269103,-97.749395,420315
2,0,2010-10-17T23:42:03Z,30.255731,-97.763386,316637
3,0,2010-10-17T19:26:05Z,30.263418,-97.757597,16516
4,0,2010-10-16T18:50:42Z,30.274292,-97.740523,5535878


In [12]:
gowalla_total['country'] = [res['cc'] for res in results]
gowalla_total['state'] = [res['admin1'] for res in results]
gowalla_total['county'] = [res['admin2'] for res in results]
gowalla_total["city_name"] = [res["name"] for res in results]

In [13]:
gowalla_total.to_csv("gowalla_total.csv")

In [ ]:
# gowalla_total = pd.read_csv("gowalla_total.csv")

In [ ]:
gowalla_us = gowalla_total[gowalla_total["country"] == "US"]
gowalla_empty_state = gowalla_us[gowalla_us["state"] == ""]
assert len(gowalla_empty_state) == 0

In [10]:
us_users_checked_in = set(gowalla_us["userid"])
number_of_us_users = len(us_users_checked_in)
print(number_of_us_users)

53362


### Handle county edge cases to match Yale county format

In [15]:
def adjust_city_of_county(row):
    split = row.split("City of")
    city_name = split[1].strip()
    new_name = city_name + " City"
    return new_name

def adjust_saint_count(row):
    return row.replace("Saint ", "St. ")

In [ ]:
connecticut_mask = gowalla_us["state"] == "Connecticut"
gowalla_us.loc[connecticut_mask, "county"] = gowalla_us.loc[connecticut_mask, "city_name"].map(town_to_planning_region)

# handle edge case for Washington D.C
washington_dc_mask = gowalla_us["city_name"] == "Washington, D.C."
gowalla_us.loc[washington_dc_mask, "county"] = "District of Columbia"
gowalla_us.loc[washington_dc_mask, "state"] = "District of Columbia"

# handle edge case for New York City
nyc_mask = ((gowalla_us["county"] == "")  | gowalla_us["county"].isna()) & (gowalla_us["city_name"] == "New York City")
gowalla_us.loc[nyc_mask, "county"] = "New York County"

## ex: City of Charlottesville is Charlottesville City in Yale Data
city_of_mask = gowalla_us['county'].str.startswith("City of")
gowalla_us.loc[city_of_mask, 'county'] = gowalla_us.loc[city_of_mask, 'county'].progress_apply(adjust_city_of_county)

# Saint -> St.
saint_mask = gowalla_us['county'].str.startswith("Saint")
gowalla_us.loc[saint_mask, 'county'] = gowalla_us.loc[saint_mask, 'county'].progress_apply(adjust_saint_count)

# ene mask
ene_mask = gowalla_us["county"] == "Dona Ana County"
gowalla_us.loc[ene_mask, "county"] = "Do\u00F1a Ana County"

# missing county (Bronx -> Bronx County)
missing_county_mask = gowalla_us["county"] == "Bronx"
gowalla_us.loc[missing_county_mask, "county"] = "Bronx County"

# desoto county
de_soto_mask = gowalla_us["county"] == "De Soto County"
gowalla_us.loc[de_soto_mask, "county"] = "DeSoto County"

bedford_mask = (gowalla_us["county"] == "Bedford City") & (gowalla_us["state"] == "Virginia")
gowalla_us.loc[bedford_mask, "county"] = "Bedford County"


0it [00:00, ?it/s]
100%|██████████| 34751/34751 [00:00<00:00, 925213.33it/s]


In [17]:
assert len(gowalla_us[(gowalla_us["county"] == "")  | (gowalla_us["county"].isna())]) == 0 # should be empty
assert len(gowalla_us[(gowalla_us["state"] == "")  | (gowalla_us["state"].isna())]) == 0 # should be empty

### Set the county for each user by the most frequent location

In [18]:
gowalla_with_county_state_count = gowalla_us.groupby(['userid','county', 'state']).size().reset_index(name='Count')
max_counts = gowalla_with_county_state_count.groupby('userid')['Count'].transform('max')
filtered_gowalla = gowalla_with_county_state_count[gowalla_with_county_state_count['Count'] == max_counts]

In [19]:
assert len(set(filtered_gowalla["userid"])) == number_of_us_users # should be 53362
filtered_gowalla.to_csv("gowalla_with_county_state.csv")

# Filter edges to remove users that didn't check in

In [20]:
filtered_edges = []
with open("gowalla_data/Gowalla_edges.txt") as f:
    for i, line in enumerate(f):
        edge = line.strip().split()
        edge[0] = int(edge[0])
        edge[1] = int(edge[1])
        if (edge[0] in us_users_checked_in and edge[1] in us_users_checked_in):
            filtered_edges.append([edge[0], edge[1]])

with open("filtered_edge_list.txt", "w") as f:
    for u, v in filtered_edges:
        f.write(f"{u} {v}\n")


# Create network graph

In [21]:
G1 = nx.read_edgelist("filtered_edge_list.txt", create_using = nx.Graph(), nodetype=int)

In [22]:
mismatch = set()
for node in G1:
    matches = filtered_gowalla[filtered_gowalla['userid'] == node]
    if not matches.empty:
        user_county = matches["county"].iloc[0]
        user_state =  matches["state"].iloc[0]

        G1.nodes[node]['county'] = user_county
        G1.nodes[node]['state'] = user_state
        yale_match = yale_2024_county[
            (yale_2024_county["state"] == user_state) & 
            (yale_2024_county["county"] == user_county)
        ]    
  
        if yale_match.empty:
            raise Exception(f"Could not find county/state pair {user_county} - {user_state} in Yale dataset")
        
        for col in yale_2024_county.columns:
            if col != "state" and col != "county":
                G1.nodes[node][col] = yale_match[col].iloc[0]

In [23]:
with open('network_graph.pickle', 'wb') as f:
    pickle.dump(G1, f, pickle.HIGHEST_PROTOCOL)